In [ ]:


import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from skimage.segmentation import slic

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, random_split
from torchvision import models, transforms

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


DATASET_DIR = "datasets"
OUTPUT_DIR = "outputs"

IMG_SIZE = 224
NUM_SUPERPIXELS = 75
K_NEIGHBORS = 5

BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 1e-4
TRAIN_SPLIT = 0.8
SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_LABELS = [
    "Automatic rifle",
    "Invalid input",
    "Knife Wound",
    "No wound found",
    "Revolver Wound",
    "Screwdriver Wound",
    "Shotgun Wound"
]
NUM_CLASSES = len(CLASS_LABELS)
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_LABELS)}

os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Using device: {DEVICE}")


IMAGE_TRANSFORM = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def build_graph(image_bgr):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_rgb = cv2.resize(image_rgb, (IMG_SIZE, IMG_SIZE))

    segments = slic(
        image_rgb,
        n_segments=NUM_SUPERPIXELS,
        compactness=10,
        start_label=0
    )

    num_nodes = segments.max() + 1


    for i in range(num_nodes):
        mask = segments == i

        pixels = image_rgb[mask]
        if len(pixels) == 0:
            mean_rgb = np.array([0, 0, 0], dtype=np.float32)
            center = (0.0, 0.0)
        else:
            mean_rgb = pixels.mean(axis=0) / 255.0
            coords = np.argwhere(mask)
            cy, cx = coords.mean(axis=0)
            center = (cx, cy)

        node_features.append(mean_rgb.astype(np.float32))
        centers.append(center)

    node_features = np.array(node_features, dtype=np.float32)

    # Build edges using nearest neighbors
    edges = []

    for i in range(num_nodes):
        xi, yi = centers[i]
        distances = []

        for j in range(num_nodes):
            if i == j:
                continue

            xj, yj = centers[j]
            d = (xi - xj) ** 2 + (yi - yj) ** 2
            distances.append((d, j))

        distances.sort(key=lambda x: x[0])

        for _, j in distances[:K_NEIGHBORS]:
            edges.append([i, j])

    if len(edges) == 0:
        edges = [[0, 0]]

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    x = torch.tensor(node_features, dtype=torch.float)

    return x, edge_index

class WoundGraphDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []

        for class_name in CLASS_LABELS:
            folder = os.path.join(root_dir, class_name)

            if not os.path.isdir(folder):
                print(f"Warning: Folder not found: {folder}")
                continue

            label = CLASS_TO_IDX[class_name]

            for fname in os.listdir(folder):
                if fname.lower().endswith(
                    (".jpg", ".jpeg", ".png", ".bmp", ".webp")
                ):
                    path = os.path.join(folder, fname)
                    self.samples.append((path, label))

        print(f"Total images found: {len(self.samples)}")

        if len(self.samples) == 0:
            raise ValueError("No images found in dataset.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        image = cv2.imread(path)
        if image is None:
            raise ValueError(f"Unable to read image: {path}")

        # CNN input
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        cnn_tensor = IMAGE_TRANSFORM(rgb)

        # Graph input
        x, edge_index = build_graph(image)

        data = Data(
            x=x,
            edge_index=edge_index,
            y=torch.tensor(label, dtype=torch.long)
        )

        # Attach image tensor for CNN branch
        data.image = cnn_tensor

        return data


class CNNFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()

        weights = models.EfficientNet_B0_Weights.DEFAULT
        backbone = models.efficientnet_b0(weights=weights)

        self.features = backbone.features

        # Freeze CNN backbone initially
        for param in self.features.parameters():
            param.requires_grad = False

    def forward(self, x):
        x = self.features(x)             # [B, 1280, 7, 7]
        x = F.adaptive_avg_pool2d(x, 1)  # [B, 1280, 1, 1]
        x = x.flatten(1)                 # [B, 1280]
        return x


class HybridCNNGNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # CNN branch
        self.cnn = CNNFeatureExtractor()
        self.cnn_fc = nn.Sequential(
            nn.Linear(1280, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # GNN branch
        self.gat1 = GATConv(3, 64, heads=4, concat=True)   # 3 -> 256
        self.gat2 = GATConv(256, 128, heads=1, concat=True)

        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(256 + 128, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

class HybridCNNGNN(nn.Module):
    def __init__(self, num_classes):
        super(HybridCNNGNN, self).__init__()


        self.cnn = CNNFeatureExtractor()

        self.cnn_fc = nn.Sequential(
            nn.Linear(1280, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )


        self.gat1 = GATConv(
            in_channels=3,
            out_channels=64,
            heads=4,
            concat=True
        )  # Output: 64 * 4 = 256

        self.gat2 = GATConv(
            in_channels=256,
            out_channels=128,
            heads=1,
            concat=True
        )  # Output: 128


        self.classifier = nn.Sequential(
            nn.Linear(384, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, data):

        images = data.image

        if images.dim() == 3:
            batch_size = data.y.size(0)
            images = images.view(
                batch_size,
                3,
                IMG_SIZE,
                IMG_SIZE
            )

        images = images.to(DEVICE)


        cnn_features = self.cnn(images)
        cnn_features = self.cnn_fc(cnn_features)

  
        x = data.x
        edge_index = data.edge_index
        batch = data.batch

        x = F.relu(self.gat1(x, edge_index))
        x = F.relu(self.gat2(x, edge_index))

        # Pool node features to graph-level features
        gnn_features = global_mean_pool(x, batch)


        fused_features = torch.cat(
            [cnn_features, gnn_features],
            dim=1
        )


        logits = self.classifier(fused_features)

        return logits


def save_line_plot(values, ylabel, title, filename):
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(values) + 1), values, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300)
    plt.close()

def save_performance_plot(precisions, recalls, f1s):
    plt.figure(figsize=(8, 5))
    plt.plot(precisions, marker="o", label="Precision")
    plt.plot(recalls, marker="s", label="Recall")
    plt.plot(f1s, marker="^", label="F1 Score")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Validation Performance Metrics")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "performance_metrics.png"), dpi=300)
    plt.close()

def save_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(10, 8))
    plt.imshow(cm)
    plt.colorbar()

    plt.xticks(
        np.arange(NUM_CLASSES),
        CLASS_LABELS,
        rotation=45,
        ha="right"
    )
    plt.yticks(
        np.arange(NUM_CLASSES),
        CLASS_LABELS
    )

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Confusion Matrix")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR, "confusion_matrix.png"),
        dpi=300
    )
    plt.close()


def evaluate(model, loader):
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)

            outputs = model(batch)
            preds = outputs.argmax(dim=1)

            y_true.extend(batch.y.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())

    accuracy = accuracy_score(y_true, y_pred) * 100
    precision = precision_score(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    )
    recall = recall_score(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    )
    f1 = f1_score(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    )

    return accuracy, precision, recall, f1, y_true, y_pred

# ==========================================================
# TRAINING
# ==========================================================
def train():
    # Dataset
    dataset = WoundGraphDataset(DATASET_DIR)

    # Split dataset
    train_size = int(TRAIN_SPLIT * len(dataset))
    val_size = len(dataset) - train_size

    train_set, val_set = random_split(
        dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED)
    )

    # Loaders
    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    val_loader = DataLoader(
        val_set,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    print(f"Train samples: {len(train_set)}")
    print(f"Validation samples: {len(val_set)}")

    # Model
    model = HybridCNNGNN(NUM_CLASSES).to(DEVICE)

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    # History
    train_losses = []
    val_accuracies = []
    val_precisions = []
    val_recalls = []
    val_f1s = []

    best_accuracy = 0.0

    # Training loop
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        progress = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{EPOCHS}"
        )

        for batch in progress:
            batch = batch.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(batch)
            loss = criterion(outputs, batch.y)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            progress.set_postfix(loss=f"{loss.item():.4f}")

        avg_train_loss = running_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation
        (
            val_accuracy,
            val_precision,
            val_recall,
            val_f1,
            _,
            _
        ) = evaluate(model, val_loader)

        val_accuracies.append(val_accuracy)
        val_precisions.append(val_precision)
        val_recalls.append(val_recall)
        val_f1s.append(val_f1)

        print(
            f"\nEpoch {epoch + 1}/{EPOCHS}"
            f"\nTrain Loss : {avg_train_loss:.4f}"
            f"\nVal Acc    : {val_accuracy:.2f}%"
            f"\nPrecision  : {val_precision:.4f}"
            f"\nRecall     : {val_recall:.4f}"
            f"\nF1 Score   : {val_f1:.4f}"
        )

        # Save best model
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "class_labels": CLASS_LABELS,
                },
                os.path.join(
                    OUTPUT_DIR,
                    "best_cnn_gnn_model.pth"
                )
            )

            print("Best model saved.")

    print("\nLoading best model...")
    checkpoint = torch.load(
        os.path.join(OUTPUT_DIR, "best_cnn_gnn_model.pth"),
        map_location=DEVICE
    )
    model.load_state_dict(checkpoint["model_state_dict"])

    (
        final_accuracy,
        final_precision,
        final_recall,
        final_f1,
        y_true,
        y_pred
    ) = evaluate(model, val_loader)

    # Classification report
    report = classification_report(
        y_true,
        y_pred,
        target_names=CLASS_LABELS,
        zero_division=0
    )

    print("\n========== CLASSIFICATION REPORT ==========")
    print(report)

    with open(
        os.path.join(OUTPUT_DIR, "classification_report.txt"),
        "w",
        encoding="utf-8"
    ) as f:
        f.write(report)

    # Metrics summary
    with open(
        os.path.join(OUTPUT_DIR, "metrics_summary.txt"),
        "w",
        encoding="utf-8"
    ) as f:
        f.write(f"Accuracy: {final_accuracy:.4f}\n")
        f.write(f"Precision: {final_precision:.4f}\n")
        f.write(f"Recall: {final_recall:.4f}\n")
        f.write(f"F1 Score: {final_f1:.4f}\n")

    # Save plots
    save_confusion_matrix(y_true, y_pred)

    save_line_plot(
        train_losses,
        "Loss",
        "Training Loss",
        "training_loss.png"
    )

    save_line_plot(
        val_accuracies,
        "Accuracy (%)",
        "Validation Accuracy",
        "validation_accuracy.png"
    )

    save_performance_plot(
        val_precisions,
        val_recalls,
        val_f1s
    )

    print("\n========== FINAL RESULTS ==========")
    print(f"Best Validation Accuracy: {best_accuracy:.2f}%")
    print(f"Final Accuracy         : {final_accuracy:.4f}")
    print(f"Final Precision        : {final_precision:.4f}")
    print(f"Final Recall           : {final_recall:.4f}")
    print(f"Final F1 Score         : {final_f1:.4f}")

    print("\nFiles saved in outputs/:")
    print(" - best_cnn_gnn_model.pth")
    print(" - classification_report.txt")
    print(" - metrics_summary.txt")
    print(" - confusion_matrix.png")
    print(" - training_loss.png")
    print(" - validation_accuracy.png")
    print(" - performance_metrics.png")

# ==========================================================
# MAIN
# ==========================================================
if __name__ == "__main__":
    train()

Using device: cpu
Total images found: 274
Train samples: 219
Validation samples: 55


Epoch 1/30: 100%|██████████| 28/28 [00:45<00:00,  1.62s/it, loss=1.9320]



Epoch 1/30
Train Loss : 1.9367
Val Acc    : 36.36%
Precision  : 0.4026
Recall     : 0.3636
F1 Score   : 0.2927
Best model saved.


Epoch 2/30: 100%|██████████| 28/28 [00:44<00:00,  1.59s/it, loss=1.9996]



Epoch 2/30
Train Loss : 1.8987
Val Acc    : 47.27%
Precision  : 0.5194
Recall     : 0.4727
F1 Score   : 0.4034
Best model saved.


...


Epoch 29/30
Train Loss : 0.1619
Val Acc    : 98.18%
Precision  : 0.9844
Recall     : 0.9818
F1 Score   : 0.9815


Epoch 30/30: 100%|██████████| 28/28 [00:42<00:00,  1.53s/it, loss=0.5032]



Epoch 30/30
Train Loss : 0.1662
Val Acc    : 98.18%
Precision  : 0.9844
Recall     : 0.9818
F1 Score   : 0.9815

Loading best model...


C:\Users\vishw\AppData\Local\Temp\ipykernel_16228\513440514.py:579: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(



========== CLASSIFICATION REPORT ==========
                   precision    recall  f1-score   support

  Automatic rifle       1.00      1.00      1.00         8
    Invalid input       1.00      1.00      1.00        11
      Knife Wound       1.00      1.00      1.00        11
   No wound found       1.00      1.00      1.00         6
   Revolver Wound       1.00      1.00      1.00         8
Screwdriver Wound       0.86      1.00      0.92         6
    Shotgun Wound       1.00      0.80      0.89         5

         accuracy                           0.98        55
        macro avg       0.98      0.97      0.97        55
     weighted avg       0.98      0.98      0.98        55


========== FINAL RESULTS ==========
Best Validation Accuracy: 98.18%
Final Accuracy         : 98.1818
Final Precision        : 0.9844
Final Recall           : 0.9818
Final F1 Score         : 0.9815

Files saved in outputs/:
 - best_cnn_gnn_model.pth
 - classification_report.txt
 - metrics_summary.txt


In [17]:

import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from skimage.segmentation import slic

from torch_geometric.data import Data
from torch_geometric.nn import GATConv, global_mean_pool

# ==========================================================
# CONFIGURATION
# ==========================================================
IMAGE_PATH = "datasets/Knife Wound/1 - Copy - Copy.jpg"
MODEL_PATH = "outputs/best_cnn_gnn_model.pth"

IMG_SIZE = 224
NUM_SUPERPIXELS = 75
K_NEIGHBORS = 5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_LABELS = [
    "Automatic rifle",
    "Invalid input",
    "Knife",
    "No wound found",
    "Revolver",
    "Screwdriver",
    "Shotgun"
]

# ==========================================================
# IMAGE TRANSFORM
# ==========================================================
IMAGE_TRANSFORM = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ==========================================================
# GRAPH CONSTRUCTION
# ==========================================================
def build_graph(image_bgr):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_rgb = cv2.resize(image_rgb, (IMG_SIZE, IMG_SIZE))

    segments = slic(
        image_rgb,
        n_segments=NUM_SUPERPIXELS,
        compactness=10,
        start_label=0
    )

    num_nodes = segments.max() + 1

    node_features = []
    centers = []

    for i in range(num_nodes):
        mask = segments == i
        pixels = image_rgb[mask]

        if len(pixels) == 0:
            mean_rgb = np.array([0, 0, 0], dtype=np.float32)
            center = (0.0, 0.0)
        else:
            mean_rgb = pixels.mean(axis=0) / 255.0
            coords = np.argwhere(mask)
            cy, cx = coords.mean(axis=0)
            center = (cx, cy)

        node_features.append(mean_rgb.astype(np.float32))
        centers.append(center)

    node_features = np.array(node_features, dtype=np.float32)

    edges = []
    for i in range(num_nodes):
        xi, yi = centers[i]
        distances = []

        for j in range(num_nodes):
            if i == j:
                continue

            xj, yj = centers[j]
            d = (xi - xj) ** 2 + (yi - yj) ** 2
            distances.append((d, j))

        distances.sort(key=lambda x: x[0])

        for _, j in distances[:K_NEIGHBORS]:
            edges.append([i, j])

    if len(edges) == 0:
        edges = [[0, 0]]

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    x = torch.tensor(node_features, dtype=torch.float)

    return x, edge_index

# ==========================================================
# CNN FEATURE EXTRACTOR
# ==========================================================
class CNNFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()

        weights = models.EfficientNet_B0_Weights.DEFAULT
        backbone = models.efficientnet_b0(weights=weights)

        self.features = backbone.features

        for param in self.features.parameters():
            param.requires_grad = False

    def forward(self, x):
        x = self.features(x)
        x = F.adaptive_avg_pool2d(x, 1)
        x = x.flatten(1)
        return x

# ==========================================================
# HYBRID CNN + GNN MODEL
# ==========================================================
class HybridCNNGNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.cnn = CNNFeatureExtractor()

        self.cnn_fc = nn.Sequential(
            nn.Linear(1280, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.gat1 = GATConv(3, 64, heads=4, concat=True)
        self.gat2 = GATConv(256, 128, heads=1, concat=True)

        self.classifier = nn.Sequential(
            nn.Linear(384, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, data):
        # Image tensor
        images = data.image

        # Single image shape: [3, H, W]
        if images.dim() == 3:
            images = images.unsqueeze(0)

        images = images.to(DEVICE)

        # CNN branch
        cnn_features = self.cnn(images)
        cnn_features = self.cnn_fc(cnn_features)

        # GNN branch
        x = data.x.to(DEVICE)
        edge_index = data.edge_index.to(DEVICE)

        # Single graph => batch vector of zeros
        if not hasattr(data, "batch") or data.batch is None:
            batch = torch.zeros(
                x.size(0),
                dtype=torch.long,
                device=DEVICE
            )
        else:
            batch = data.batch.to(DEVICE)

        x = F.relu(self.gat1(x, edge_index))
        x = F.relu(self.gat2(x, edge_index))
        gnn_features = global_mean_pool(x, batch)

        # Fusion
        fused = torch.cat(
            [cnn_features, gnn_features],
            dim=1
        )

        # Classification
        logits = self.classifier(fused)

        return logits

# ==========================================================
# PREDICTION FUNCTION
# ==========================================================
def predict(image_path):
    # Load checkpoint
    checkpoint = torch.load(
        MODEL_PATH,
        map_location=DEVICE
    )

    # Load model
    model = HybridCNNGNN(
        num_classes=len(CLASS_LABELS)
    ).to(DEVICE)

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )
    model.eval()

    # Read image
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(
            f"Unable to read image: {image_path}"
        )

    # CNN input
    rgb = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )
    cnn_tensor = IMAGE_TRANSFORM(rgb)

    # Graph input
    x, edge_index = build_graph(image)

    # Create graph data
    data = Data(
        x=x,
        edge_index=edge_index
    )
    data.image = cnn_tensor

    # Predict
    with torch.no_grad():
        outputs = model(data)
        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        confidence, pred_idx = torch.max(
            probabilities,
            dim=1
        )

    predicted_class = CLASS_LABELS[
        pred_idx.item()
    ]
    confidence_percent = (
        confidence.item() * 100
    )

    # Print all probabilities
    print("\nClass Probabilities:")
    for i, label in enumerate(CLASS_LABELS):
        prob = probabilities[0, i].item() * 100
        print(f"{label:20s}: {prob:.2f}%")

    print("\n==============================")
    print(f"Predicted Class : {predicted_class}")
    print(f"Confidence      : {confidence_percent:.2f}%")
    print("==============================")

    return predicted_class, confidence_percent

# ==========================================================
# MAIN
# ==========================================================
if __name__ == "__main__":
    predict(IMAGE_PATH)

C:\Users\vishw\AppData\Local\Temp\ipykernel_16228\1660080927.py:205: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(



Class Probabilities:
Automatic rifle     : 0.10%
Invalid input       : 0.89%
Knife               : 92.63%
No wound found      : 1.01%
Revolver            : 0.81%
Screwdriver         : 2.19%
Shotgun             : 2.37%

Predicted Class : Knife
Confidence      : 92.63%
